# Stage 11 V2d — Canonical CPU Notehead Bridge

This is the focused follow-up to the completed V2d benchmark. It does **not** run Oemer again and does **not** need GPU. It re-executes only the frozen Restore model for the 18 teacher-present notehead pages under the committed canonical CPU profile, proves pixel identity against V2c canonical evidence, and reuses only SHA/fingerprint-validated V2d detector records.


In [ ]:
# 1) Exact canonical CPU runtime. Fail before any expensive work if Colab changed.
import platform, subprocess, sys
print('python', platform.python_version())
if platform.python_version() != '3.13.5':
    raise RuntimeError('Canonical evidence requires Python 3.13.5 exactly; stop without running Restore.')
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torch', 'torchvision', 'torchaudio'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.10.0', '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True)
verify = r'''import platform, torch, cv2, numpy as np
print('python', platform.python_version())
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('opencv', cv2.__version__)
print('numpy', np.__version__)
assert platform.python_version() == '3.13.5'
assert torch.__version__ == '2.10.0+cpu'
assert not torch.cuda.is_available()
'''
subprocess.run([sys.executable, '-c', verify], check=True)
print('CANONICAL CPU RUNTIME READY')


In [ ]:
# 2) Mount the existing crash-safe Drive cache and refresh the development branch source.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil, subprocess, sys
CACHE = Path('/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_EVAL/V2D_CACHE_V1')
RESULTS = Path('/content/drive/MyDrive/ST_SCORE_RESTORE_STAGE11_EVAL/V2D_RESULTS')
for required in [CACHE/'cache_manifest.json', CACHE/'exact_inputs'/'v2a_candidate_512.torchscript.pt', RESULTS/'v2d_colab_gpu_detector_benchmark_result.json']:
    if not required.exists():
        raise FileNotFoundError(required)
REPO = Path('/content/st-score-restore-engine')
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b','stage11-v2c-semantic-detector-corpus-expansion','https://github.com/khfy7wpr5p-maker/st-score-restore-engine.git',str(REPO)], check=True)
sys.path.insert(0, str(REPO/'src'))
print('DRIVE CACHE + REPO READY')


In [ ]:
# 3) Run the focused bridge in a fresh process. No Oemer/ONNX inference is repeated.
import os, subprocess, sys
LOG = RESULTS / 'v2d_canonical_cpu_notehead_bridge.log'
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['PYTHONFAULTHANDLER'] = '1'
env['PYTHONPATH'] = str(REPO/'src') + os.pathsep + env.get('PYTHONPATH','')
cmd = [sys.executable, '-m', 'st_score_restore.stage11_v2d_canonical_cpu_notehead_bridge', '--run']
print('running:', ' '.join(cmd))
print('log:', LOG)
with LOG.open('a', encoding='utf-8', buffering=1) as log_handle:
    log_handle.write('\n===== CANONICAL CPU NOTEHEAD BRIDGE RUN =====\n')
    proc = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
        log_handle.write(line)
    rc = proc.wait()
if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)


## Success markers

`CANONICAL CPU RUNTIME READY` → `DRIVE CACHE + REPO READY` → `CANONICAL CPU PREFLIGHT PASS` → `CANONICAL CPU NOTEHEAD BRIDGE PASS` → `SAVED:`

This bridge intentionally keeps `semanticPreservationEstablished=false`, production disabled, and Stage 12 closed. It upgrades only the runtime provenance of the selected notehead evidence if pixel identity passes on all 18 pages.
